# Namenslisten gegen die Graphen prüfen

Wie gut deckt eine externe Namensliste einen Wissensgraphen ab? Zwei Zahlen,
die Verschiedenes messen und nicht ineinander umrechenbar sind:

| | misst | Konsequenz |
|---|---|---|
| **`titles_matched`** | Anteil der *Einträge* mit Entsprechung im Graphen | **Kosten**: `1/titles_matched` Ziehungen je Treffer |
| **`nodes_covered`** | Anteil der *Knoten*, den die Liste erreicht (bezogen auf \|V\|) | **Gültigkeit**: mehr kann eine Ziehung daraus nie sehen |

Die Trefferquote lässt sich durch mehr Ziehungen erkaufen, die Abdeckung nicht.
Für den Zufallssprung von DURW ist deshalb die zweite die entscheidende — sie
ist die Obergrenze dessen, was ein so emulierter Sprung erreichen kann.

Läuft aus `Code/`. Unterbau: `namelists.py` (Quellen und Auswertung),
`title_overlap.py` (Normalisierung), `build_name_index.py` (Ziehungsindizes).

In [ ]:
import importlib

import pandas as pd

import config
import namelists
import title_overlap
importlib.reload(title_overlap); importlib.reload(namelists)

from graphs import loader
from namelists import NameList, SOURCES, head, name_lookup, overlap
from title_overlap import VARIANTS, Normalizer

pd.set_option("display.width", 200)

for k, v in SOURCES.items():
    flag = "realizable" if v.realizable else "COMPARISON (aus dem Graphen)"
    print(f"{k:16s} {v.path.name:44s} {flag}")

## Der Baukasten

`lookup` ist die teure Struktur: ein dict über alle normalisierten Knotennamen,
bei 6,5 Mio Knoten rund 26 s und ein bis zwei GB. Er hängt nur am **Normalizer**,
nicht an der Liste — deshalb einmal je (Graph, Normalizer) bauen und an alle
`overlap()`-Aufrufe weiterreichen. Jeder weitere Anteil kostet dann fast nichts.

In [ ]:
def evaluate(graph_name, jobs, log=True):
    """jobs: Liste von (Beschriftung, NameList) -> DataFrame.

    Gruppiert nach Normalizer, damit der lookup je Normalizer nur einmal
    entsteht."""
    loader.clear_cache()
    g = loader.load_graph(graph_name)
    rows, lookups = [], {}
    for label, nl in jobs:
        if nl.norm not in lookups:
            if log: print(f"  lookup für {nl.norm.label()} ...", flush=True)
            lookups[nl.norm] = name_lookup(g, nl.norm)
        r = overlap(g, nl, lookup=lookups[nl.norm])
        rows.append({"liste": label, "Einträge": r["n_entries"],
                     "Treffer %": round(r["titles_matched"] * 100, 2),
                     "Abdeckung %": round(r["nodes_covered"] * 100, 3),
                     "Knoten": r["n_nodes_hit"],
                     "Zieh./Treffer": round(r["draws_per_hit"], 1)})
        if log: print(f"  {label:28s} Treffer {r['titles_matched']:7.2%}  "
                      f"Abdeckung {r['nodes_covered']:7.3%}", flush=True)
    del lookups
    return pd.DataFrame(rows).set_index("liste")

## Anteile beliebig festlegen

`head(quelle, fraction=…)` bzw. `head(quelle, n=…)` schneidet eine **sortierte**
Liste vorne ab: `top-q` ist nach QRank geordnet, die In-Grad-Listen nach
Eingangsgrad. Bei den alphabetischen Wikipedia-Dumps warnt die Funktion, weil ein
Präfix dort keine „Top-n" wäre, sondern alles, was mit `!` und `A` anfängt.

`FRACTIONS` ist der einzige Knopf.

In [ ]:
FRACTIONS = [0.01, 0.05, 0.1, 0.25, 0.5, 1.0]

def fraction_jobs(source, fractions=FRACTIONS):
    return [(f"{source} {f:>6.1%}", head(source, fraction=f)) for f in fractions]

GRAPH = "gpt4_io"          # oder "gpt4o_io"

jobs = (fraction_jobs("top-q")
        + fraction_jobs("indeg-gpt4_io")      # eigene Liste = Obergrenze
        + fraction_jobs("indeg-gpt4o_io"))    # fremde Liste = echter Test

df = evaluate(GRAPH, jobs)
df

## Beide Graphen nebeneinander

**Zu den In-Grad-Listen:** sie stammen aus dem Graphen selbst
(`realizable=False`). Gegen den *eigenen* Graphen ist die Trefferquote per
Konstruktion 100 % und die Abdeckung genau der genommene Anteil — das ist die
Obergrenze, kein Verfahren. Die eigentliche Frage ist der **Kreuztest**: wie weit
deckt die Prominenzliste des einen Graphen den anderen ab?

In [ ]:
both = {}
for graph in ("gpt4_io", "gpt4o_io"):
    both[graph] = evaluate(graph, jobs, log=False)
    print(f"{graph} fertig", flush=True)

pd.concat(both, names=["graph"])[["Treffer %", "Abdeckung %", "Zieh./Treffer"]]

## Vollständige Wikipedia-Dumps

Ohne Anteil, weil alphabetisch sortiert — ein Präfix wäre keine sinnvolle
Teilmenge. Rechnet einige Minuten (enwiki hat 19,2 Mio Einträge).

In [ ]:
wiki_jobs = [("enwiki", SOURCES["enwiki"]), ("dewiki", SOURCES["dewiki"])]
pd.concat({g: evaluate(g, wiki_jobs, log=False) for g in ("gpt4_io", "gpt4o_io")},
          names=["graph"])

## Eine neue Datei anschließen

Für eine Liste, die noch nicht in `SOURCES` steht, reicht ein `NameList`. Danach
in `namelists.SOURCES` eintragen, wenn sie dauerhaft bleiben soll — erst dann
entstehen Registry-Einträge, und erst dann ist ein Ziehungsindex baubar:

    python build_name_index.py --graphs gpt4_io --sources <name>

In [ ]:
# Zeilenliste, ein Name je Zeile:
eigene = NameList(config.ADDITIONALS_DIR / "meine_namen.txt",
                  VARIANTS["+whitespace"])

# Tabelle ohne Kopfzeile, Name in Spalte 1, Tab-getrennt (wie die In-Grad-Listen):
eigene_tsv = NameList(config.ADDITIONALS_DIR / "meine.tsv",
                      VARIANTS["+whitespace"],
                      columns=(1,), delimiter="\t", skip_header=False)

# Tabelle mit Kopfzeile, mehrere Namensspalten in Prioritätsreihenfolge
# (erster Treffer gewinnt, gezogen wird pro Eintrag -- nicht pro Name):
eigene_csv = NameList(config.ADDITIONALS_DIR / "meine.csv",
                      Normalizer(nfc=True, underscores_to_spaces=False,
                                 casefold=True, collapse_whitespace=False),
                      columns=("enwiki_title", "label"))

# evaluate(GRAPH, [("meine Liste", eigene)])